In [0]:
%sql
create catalog if not exists olist_brazilian_ecommerce ;  

In [0]:
%sql
create schema if not exists olist_brazilian_ecommerce.source ;  

create volume if not exists olist_brazilian_ecommerce.source.source ;  

In [0]:
%python
from pyspark.sql.functions import *
from pyspark.sql.types import *

from pyspark.sql import *

In [0]:
import sys
import importlib

sys.path.append(
    "/Workspace/Users/yousef.abdelrahman.09@gmail.com/olist_brazilian_ecommerce_project/src"
)

from utils import exploration_utils

importlib.reload(exploration_utils)

profile_nulls = exploration_utils.profile_nulls
check_duplicates = exploration_utils.check_duplicates

# explore data 

## customers

In [0]:
customers = spark.read.csv("/Volumes/olist_brazilian_ecommerce/source/source/archive/olist_customers_dataset.csv", header=True, inferSchema=True)
display(customers)



In [0]:
customers.printSchema()

In [0]:
states = spark.read.csv("/Volumes/olist_brazilian_ecommerce/source/source/archive/states.csv", header=True, inferSchema=True)
true_states = states.select(col("UF")).show()


In [0]:
flase_state =customers.count() - customers.join(states, customers["customer_state"] == states["UF"]).count()
print(f"There are {flase_state} false states")

In [0]:
customers.createOrReplaceTempView("customers")

In [0]:
customers.withColumn("customer_zip_code_prefix_len", length(customers["customer_zip_code_prefix"])) \
         .agg(
             min("customer_zip_code_prefix_len").alias("min_len"), 
             max("customer_zip_code_prefix_len").alias("max_len")
         ).show()

In [0]:
%sql
select 
    COUNT(DISTINCT customer_unique_id)as total_customers_unique ,
    COUNT( customer_id)as total_customer_records
from customers;

In [0]:
%sql
select count(distinct customer_state) as tottal_state
 from customers ; 


In [0]:
%sql
select 
    customer_state , 
    customer_city , 
    COUNT(*) as total_customers_in_city
from customers 
group by  customer_state ,customer_city ; 

In [0]:
%sql
SELECT COUNT(DISTINCT customer_city) AS total_cities
FROM customers;
-- theere are same cities name but diff states

In [0]:

display(profile_nulls(customers))

In [0]:
%sql
SELECT count(*)-COUNT(DISTINCT *) AS total_dupp
FROM customers;

In [0]:
%sql
SELECT
    COUNT(*) - COUNT(DISTINCT customer_id, customer_unique_id, customer_zip_code_prefix,
                       customer_city, customer_state) AS duplicate_rows
FROM customers;

## orders_dataset

In [0]:
orders_dataset = spark.read.csv("/Volumes/olist_brazilian_ecommerce/source/source/archive/olist_orders_dataset.csv", header=True, inferSchema=True) 
display(orders_dataset)



In [0]:
orders_dataset.printSchema()

In [0]:
orders_dataset.createOrReplaceTempView("orders_dataset")

In [0]:

display(profile_nulls(orders_dataset))

In [0]:
orders_dataset.filter(col("order_approved_at").isNull()).show()

In [0]:
%sql
SELECT DISTINCT(order_status) FROM orders_dataset;

In [0]:
%sql
SELECT
    order_status,
    COUNT(*) AS total_orders,
    SUM(CASE WHEN order_approved_at IS NULL THEN 1 ELSE 0 END) AS null_approved,
    ROUND(
        SUM(CASE WHEN order_approved_at IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        3
    ) AS null_percentage
FROM orders_dataset
GROUP BY order_status
ORDER BY null_approved DESC;

In [0]:
count_orders_dataset = orders_dataset.count()
print(count_orders_dataset)

In [0]:
%sql
SELECT 
    order_status,
    COUNT(*) AS order_count,
    ROUND(
        COUNT(*) / SUM(COUNT(*)) OVER () * 100,
        3
    ) AS percentage
FROM orders_dataset
GROUP BY order_status
ORDER BY order_count;

## ### order_items_dataset


In [0]:
order_items_dataset = spark.read.csv("/Volumes/olist_brazilian_ecommerce/source/source/archive/olist_order_items_dataset.csv",header=True, inferSchema=True )
order_items_dataset.show()

In [0]:
order_items_dataset.printSchema()

In [0]:
order_items_dataset.createOrReplaceTempView("order_items_dataset")

In [0]:
display(profile_nulls(order_items_dataset))

In [0]:
%sql
select   
    order_item_id , 
    count(order_item_id)  as freq , 
    count(order_item_id) / SUM(COUNT(*)) OVER () * 100 as percentage
from order_items_dataset 
group by order_item_id 
order by order_item_id


In [0]:
display(check_duplicates(order_items_dataset , ["order_id","order_item_id"  ]))

In [0]:
order_items_dataset.agg(
    min("price").alias("min_price"),
    max("price").alias("max_price"),
    avg("price").alias("avg_price")
).show()

In [0]:
order_items_dataset.agg(
    min("freight_value").alias("min_freight_value"),
    max("freight_value").alias("max_freight_value"),
    avg("freight_value").alias("avg_freight_value")
).show()

## products_dataset

In [0]:
/Volumes/olist_brazilian_ecommerce/source/source/archive/olist_products_dataset.csv

In [0]:
products_dataset = spark.read.csv("/Volumes/olist_brazilian_ecommerce/source/source/archive/olist_products_dataset.csv",header=True, inferSchema=True )
products_dataset.show()


In [0]:
products_dataset.printSchema()

In [0]:
display(profile_nulls(products_dataset))


In [0]:
products_dataset.filter(products_dataset.product_category_name.isNull() & products_dataset.product_name_lenght.isNull() & products_dataset.product_description_lenght.isNull() & products_dataset.product_photos_qty.isNull()).show()

In [0]:
products_dataset.filter(products_dataset.product_category_name.isNull() & products_dataset.product_name_lenght.isNull() & products_dataset.product_description_lenght.isNull() & products_dataset.product_photos_qty.isNull()).count()

In [0]:
check_duplicates(products_dataset)

In [0]:
check_duplicates(products_dataset , ["product_id"])

In [0]:

numeric_cols = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

products_dataset.agg(
    *[
        min(c).alias(f"{c}_min")
        for c in numeric_cols
    ]
).show()

In [0]:
products_dataset.agg(
    *[
        max(c).alias(f"{c}_max")
        for c in numeric_cols
    ]
).show()

In [0]:
products_dataset.groupBy("product_category_name") \
    .count() \
    .orderBy(desc("count")) \
    .show(50, truncate=False)

## olist_order_payments

In [0]:
olist_order_payments = spark.read.csv("/Volumes/olist_brazilian_ecommerce/source/source/archive/olist_order_payments_dataset.csv",inferSchema= True, header=True,)

display(olist_order_payments)


In [0]:
olist_order_payments.printSchema()

In [0]:
olist_order_payments.createOrReplaceTempView("olist_order_payments")

In [0]:
profile_nulls(olist_order_payments).display()

In [0]:
check_duplicates(olist_order_payments)

In [0]:
check_duplicates(olist_order_payments, ["order_id","payment_sequential"])

In [0]:
%sql
select 
    payment_sequential , 
    count(payment_sequential) as freq
from olist_order_payments 
group by payment_sequential
order by payment_sequential

In [0]:
%sql
select 
    payment_installments , 
    count(payment_installments) as freq
from olist_order_payments 
group by payment_installments
order by payment_installments

In [0]:
%sql
SELECT *
FROM olist_order_payments
WHERE payment_installments = 0;

In [0]:
%sql
select 
    payment_type , 
    count(payment_type) as freq_per_type ,
    ROUND(count(payment_type) / SUM(COUNT(*)) OVER () * 100 , 2 )as percentage
from olist_order_payments 
group by payment_type
order by percentage desc

In [0]:

olist_order_payments.agg(
    min("payment_value").alias("min_payment_value"),
    max("payment_value").alias("max_payment_value"),
    avg("payment_value").alias("avg_payment_value")
).show()

In [0]:
%sql
SELECT order_id
FROM olist_order_payments
GROUP BY order_id
HAVING MIN(payment_sequential) != 1
    OR MAX(payment_sequential) != COUNT(*);

In [0]:
%sql
SELECT
    order_id,
    COUNT(*) AS payment_count,
    MIN(payment_sequential) AS min_seq,
    MAX(payment_sequential) AS max_seq,
    SORT_ARRAY(COLLECT_LIST(payment_sequential)) AS sequences
FROM olist_order_payments
GROUP BY order_id
HAVING MIN(payment_sequential) != 1
    OR MAX(payment_sequential) != COUNT(*)
ORDER BY order_id;

In [0]:
%sql
SELECT *
FROM olist_order_payments
WHERE order_id = '00ac05fe0fc047c54418098eb64e3aaa';

## order_reviews

In [0]:
order_reviews = spark.read.csv(
    "/Volumes/olist_brazilian_ecommerce/source/source/archive/olist_order_reviews_dataset.csv",
    header=True,
    inferSchema=True
)

display(order_reviews)

In [0]:
order_reviews.printSchema()

In [0]:
order_reviews.createOrReplaceTempView("order_reviews")

In [0]:
profile_nulls(order_reviews).display()

In [0]:
check_duplicates(order_reviews)

In [0]:
check_duplicates(order_reviews , ["order_id", "review_id"])


In [0]:
%sql
SELECT
    order_id,
    review_id,
    COUNT(*) AS frequency
FROM order_reviews
GROUP BY order_id, review_id
HAVING COUNT(*) > 1
ORDER BY frequency DESC;

In [0]:
order_review = (
    spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .option("mode", "PERMISSIVE")
    .csv("/Volumes/olist_brazilian_ecommerce/source/source/archive/olist_order_reviews_dataset.csv")
)

display(order_review)

In [0]:
order_review.createOrReplaceTempView("order_review")

In [0]:
%sql
SELECT
    order_id,
    review_id,
    COUNT(*) AS frequency
FROM order_review
GROUP BY order_id, review_id
HAVING COUNT(*) > 1
ORDER BY frequency DESC;

In [0]:
check_duplicates(order_review)

In [0]:
check_duplicates(order_review)

check_duplicates(order_review, ["order_id", "review_id"])

In [0]:
from pyspark.sql import functions as F

order_review.select(
    F.count("*").alias("total_rows"),
    F.count("order_id").alias("non_null_order_id"),
    F.count("review_id").alias("non_null_review_id")
).show()

In [0]:
print("Original read:", order_reviews.count())
print("New read:", order_review.count())

In [0]:
%sql 
select review_score
from order_review 
group by review_score; 

In [0]:
profile_nulls(order_review).display()

In [0]:
order_review.printSchema()

In [0]:
order_review.createOrReplaceTempView("order_review")

In [0]:
%sql 
select review_score ,
 count(review_score) as freq_per_score , 
 round(count(review_score)  / sum(count(*)) over()  * 100 ,2) as freq_per_score_percentage
from order_review
group by review_score
order by  review_score desc



# olist_sellers

In [0]:
olist_sellers  =  spark.read.csv("/Volumes/olist_brazilian_ecommerce/source/source/archive/olist_sellers_dataset.csv", header=True, inferSchema=True)
olist_sellers.createOrReplaceTempView("olist_sellers")
olist_sellers.display()

In [0]:
olist_sellers.printSchema()

In [0]:
profile_nulls(olist_sellers).display()

In [0]:
check_duplicates(olist_sellers)

In [0]:
olist_sellers.agg(
    max(col("seller_zip_code_prefix")).alias("max_seller_zip_code_prefix"),
    min(col("seller_zip_code_prefix")).alias("min_seller_zip_code_prefix")).display()

## product_category_name_translation

In [0]:
/Volumes/olist_brazilian_ecommerce/source/source/archive/product_category_name_translation.csv

In [0]:
product_category_name_translation = spark.read.csv("/Volumes/olist_brazilian_ecommerce/source/source/archive/product_category_name_translation.csv", header=True, inferSchema=True)
product_category_name_translation.createOrReplaceTempView("product_category_name_translation")
product_category_name_translation.display()

In [0]:
product_category_name_translation.printSchema()

In [0]:
check_duplicates(product_category_name_translation)

In [0]:
profile_nulls(product_category_name_translation).display()

## geolocation_dataset

In [0]:
/Volumes/olist_brazilian_ecommerce/source/source/archive/olist_geolocation_dataset.csv

In [0]:
geolocation_dataset = spark.read.csv("/Volumes/olist_brazilian_ecommerce/source/source/archive/olist_geolocation_dataset.csv", header=True, inferSchema=True)
geolocation_dataset.createOrReplaceTempView("geolocation_dataset")
geolocation_dataset.display()

In [0]:
geolocation_dataset.printSchema()

In [0]:
check_duplicates(geolocation_dataset)

In [0]:
profile_nulls(geolocation_dataset).display()

In [0]:

# Number of distinct ZIP prefixes
print(
    "Distinct ZIP prefixes:",
    geolocation_dataset.select("geolocation_zip_code_prefix")
    .distinct()
    .count()
)

In [0]:
invalid_coordinates = geolocation_dataset.filter(
    col("geolocation_lat").isNull()
    | col("geolocation_lng").isNull()
    | (~col("geolocation_lat").between(-90, 90))
    | (~col("geolocation_lng").between(-180, 180))
)

display(invalid_coordinates)